In [6]:
import math
from dataclasses import dataclass

In [7]:
# ======================================================================================
# Basic LCDM time-redshift conversion (flat LCDM: matter + Lambda)
# ======================================================================================

def H0_to_Gyr_inv(H0_km_s_Mpc: float) -> float:
    """
    Convert H0 from km/s/Mpc to 1/Gyr.
    """
    Mpc_km = 3.0856775814913673e19
    Gyr_s = 1.0e9 * 365.25 * 24.0 * 3600.0
    return (H0_km_s_Mpc / Mpc_km) * Gyr_s


def cosmic_time_Gyr_LCDM(
    z: float,
    *,
    H0_km_s_Mpc: float = 67.4,
    Omega_m: float = 0.315,
) -> float:
    """
    Cosmic time t(z) in Gyr for flat LCDM with matter + Lambda.
    Radiation neglected, which is fine for z ≲ 50.

    Exact analytic formula:
        t(a) = 2 / (3 H0 sqrt(Omega_L)) * asinh( sqrt(Omega_L/Omega_m) * a^(3/2) )
    where a = 1/(1+z).
    """
    if z < 0:
        raise ValueError("z must be >= 0")
    if not (0.0 < Omega_m < 1.0):
        raise ValueError("Omega_m must satisfy 0 < Omega_m < 1")

    Omega_L = 1.0 - Omega_m
    a = 1.0 / (1.0 + z)
    H0_Gyr_inv = H0_to_Gyr_inv(H0_km_s_Mpc)

    prefactor = 2.0 / (3.0 * H0_Gyr_inv * math.sqrt(Omega_L))
    arg = math.sqrt(Omega_L / Omega_m) * a**1.5
    return prefactor * math.asinh(arg)


def redshift_from_cosmic_time_Gyr_LCDM(
    t_Gyr: float,
    *,
    H0_km_s_Mpc: float = 67.4,
    Omega_m: float = 0.315,
) -> float:
    """
    Invert t(z) analytically to get z(t) in flat LCDM (matter + Lambda).
    """
    if t_Gyr <= 0:
        raise ValueError("t_Gyr must be > 0")
    if not (0.0 < Omega_m < 1.0):
        raise ValueError("Omega_m must satisfy 0 < Omega_m < 1")

    Omega_L = 1.0 - Omega_m
    H0_Gyr_inv = H0_to_Gyr_inv(H0_km_s_Mpc)

    x = 1.5 * H0_Gyr_inv * math.sqrt(Omega_L) * t_Gyr
    sinh_x = math.sinh(x)

    a = (math.sqrt(Omega_m / Omega_L) * sinh_x) ** (2.0 / 3.0)
    z = 1.0 / a - 1.0
    return z


def z_after_delay(
    z_start: float,
    tau_Gyr: float,
    *,
    H0_km_s_Mpc: float = 67.4,
    Omega_m: float = 0.315,
) -> float:
    """
    Starting at redshift z_start, evolve forward in cosmic time by tau_Gyr,
    and return the final redshift z_end.
    """
    if z_start < 0:
        raise ValueError("z_start must be >= 0")
    if tau_Gyr <= 0:
        raise ValueError("tau_Gyr must be > 0")

    t_start = cosmic_time_Gyr_LCDM(
        z_start, H0_km_s_Mpc=H0_km_s_Mpc, Omega_m=Omega_m
    )
    t_end = t_start + tau_Gyr

    t0 = cosmic_time_Gyr_LCDM(
        0.0, H0_km_s_Mpc=H0_km_s_Mpc, Omega_m=Omega_m
    )
    if t_end >= t0:
        raise ValueError(
            f"tau too large: resulting time t_end={t_end:.3f} Gyr exceeds age of Universe t0={t0:.3f} Gyr."
        )

    return redshift_from_cosmic_time_Gyr_LCDM(
        t_end, H0_km_s_Mpc=H0_km_s_Mpc, Omega_m=Omega_m
    )


In [8]:
# ======================================================================================
# Phenomenological monopole bound-fraction gate
# ======================================================================================

def bound_fraction(a: float, kappa: float, a_t: float) -> float:
    """
    Fraction of monopoles still behaving as cold/clustered matter:
        F(a) = (1 - a^kappa) / (1 + (a/a_t)^kappa)

    For kappa > 0:
      - at early times (a << a_t): F ~ 1
      - at late times  (a >> a_t): F ~ 0
    """
    if a <= 0:
        raise ValueError("a must be > 0")
    if a_t <= 0:
        raise ValueError("a_t must be > 0")
    if kappa <= 0:
        raise ValueError("kappa must be > 0")

    return (1.0 - a**kappa) / (1.0 + (a / a_t) ** kappa)


def solve_kappa_at_from_two_points(
    z1: float,
    z2: float,
    F1: float,
    F2: float,
):
    """
    Solve for (kappa, a_t) from:
        F(a1)=F1
        F(a2)=F2
    with
        F(a)=1 / (1 + (a/a_t)^kappa)

    Usually:
        z1 > z2
        F1 close to 1 (e.g. 0.99)
        F2 close to 0 (e.g. 0.01)
    """
    if not (z1 > z2 >= 0.0):
        raise ValueError("Need z1 > z2 >= 0")
    if not (0.0 < F1 < 1.0 and 0.0 < F2 < 1.0):
        raise ValueError("Require 0 < F1,F2 < 1")
    if F1 <= F2:
        raise ValueError("Need F1 > F2 since more monopoles should remain at z1 than at z2.")

    a1 = 1.0 / (1.0 + z1)
    a2 = 1.0 / (1.0 + z2)

    y1 = (1.0 - F1) / F1
    y2 = (1.0 - F2) / F2

    kappa = math.log(y2 / y1) / math.log(a2 / a1)
    a_t = a1 / (y1 ** (1.0 / kappa))

    return kappa, a_t


def solve_kappa_at_from_z1_tau(
    z1: float,
    tau_Gyr: float,
    F1: float = 0.99,
    F2: float = 0.01,
    *,
    H0_km_s_Mpc: float = 67.4,
    Omega_m: float = 0.315,
):
    """
    Main utility:
      1) start from z1,
      2) compute z2 by evolving forward by tau_Gyr,
      3) solve for (kappa, a_t) such that
           F(z1)=F1 and F(z2)=F2.
    """
    z2 = z_after_delay(
        z1,
        tau_Gyr,
        H0_km_s_Mpc=H0_km_s_Mpc,
        Omega_m=Omega_m,
    )

    kappa, a_t = solve_kappa_at_from_two_points(z1, z2, F1, F2)

    return MonopoleTransitionModel(
        z1=z1,
        z2=z2,
        tau_Gyr=tau_Gyr,
        F1=F1,
        F2=F2,
        kappa=kappa,
        a_t=a_t,
        z_t=1.0 / a_t - 1.0,
        H0_km_s_Mpc=H0_km_s_Mpc,
        Omega_m=Omega_m,
    )



In [9]:
# ======================================================================================
# Container for results
# ======================================================================================

@dataclass
class MonopoleTransitionModel:
    z1: float
    z2: float
    tau_Gyr: float
    F1: float
    F2: float
    kappa: float
    a_t: float
    z_t: float
    H0_km_s_Mpc: float
    Omega_m: float

    def fraction_at_z(self, z: float) -> float:
        a = 1.0 / (1.0 + z)
        return bound_fraction(a, self.kappa, self.a_t)

    def summary(self) -> str:
        t1 = cosmic_time_Gyr_LCDM(
            self.z1, H0_km_s_Mpc=self.H0_km_s_Mpc, Omega_m=self.Omega_m
        )
        t2 = cosmic_time_Gyr_LCDM(
            self.z2, H0_km_s_Mpc=self.H0_km_s_Mpc, Omega_m=self.Omega_m
        )

        return (
            f"Monopole transition model\n"
            f"--------------------------\n"
            f"z1            = {self.z1:.4f}\n"
            f"z2            = {self.z2:.4f}\n"
            f"t(z1)         = {t1:.4f} Gyr\n"
            f"t(z2)         = {t2:.4f} Gyr\n"
            f"tau           = {self.tau_Gyr:.4f} Gyr\n"
            f"F(z1)         = {self.F1:.4f}\n"
            f"F(z2)         = {self.F2:.4f}\n"
            f"kappa         = {self.kappa:.4f}\n"
            f"a_t           = {self.a_t:.6f}\n"
            f"z_t           = {self.z_t:.4f}\n"
        )


In [11]:
# ======================================================================================
# Example usage
# ======================================================================================

# Example:
# around z1=10, 99% are still CDM-like
# after tau=0.01 Gyr = 10 Myr, only 1% remain bound
model = solve_kappa_at_from_z1_tau(
z1=10.0,
tau_Gyr=1e-2,
F1=0.99,
F2=0.01,
H0_km_s_Mpc=67.4,
Omega_m=0.315,
)

print(model.summary())

# sanity checks
for z in [15, 10, model.z_t, model.z2, 2, 0]:
    print(f"z={z:8.4f} -> F_bound={model.fraction_at_z(z):.6f}")

# sanity checks
# for z in [15, 10, model.z_t, model.z2, 3.9, 0]:
#     print(f"z={z:8.4f} -> F_bound={model.fraction_at_z(z):.6f}")

Monopole transition model
--------------------------
z1            = 10.0000
z2            = 9.8473
t(z1)         = 0.4722 Gyr
t(z2)         = 0.4822 Gyr
tau           = 0.0100 Gyr
F(z1)         = 0.9900
F(z2)         = 0.0100
kappa         = 657.4583
a_t           = 0.091547
z_t           = 9.9234

z= 15.0000 -> F_bound=1.000000
z= 10.0000 -> F_bound=0.990000
z=  9.9234 -> F_bound=0.500000
z=  9.8473 -> F_bound=0.010000


OverflowError: (34, 'Numerical result out of range')